# Machine Learning Models

This notebook trains and compares multiple classification models to predict productivity estimation classes.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
df = pd.read_csv("dataset_model_ready.csv")
df.head()

X = df.drop(columns=['Delta_class'])
y = df['Delta_class']

## Feature types

We distinguish between categorical and numerical features
to apply appropriate preprocessing.

In [3]:
categorical_features = X.columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Models
We start with baseline and tree-based models, followed by more flexible classifiers.

In [5]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
    ),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM (RBF)': SVC(kernel='rbf'),
    'MLP': MLPClassifier(
        hidden_layer_sizes=(100, 50),
        max_iter=500,
        random_state=42
    )
}

In [6]:
results = []

for name, model in models.items():
    
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Macro_F1': f1_score(y_test, y_pred, average='macro')
    })
    
    print(f"\n{name}")
    print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)



Logistic Regression
                  precision    recall  f1-score   support

      Sottostima       0.00      0.00      0.00         9
      Sovrastima       0.00      0.00      0.00        28
Stima realistica       0.85      1.00      0.92       203

        accuracy                           0.85       240
       macro avg       0.28      0.33      0.31       240
    weighted avg       0.72      0.85      0.78       240


Decision Tree
                  precision    recall  f1-score   support

      Sottostima       0.00      0.00      0.00         9
      Sovrastima       0.80      0.14      0.24        28
Stima realistica       0.86      1.00      0.92       203

        accuracy                           0.86       240
       macro avg       0.55      0.38      0.39       240
    weighted avg       0.82      0.86      0.81       240


Random Forest
                  precision    recall  f1-score   support

      Sottostima       0.00      0.00      0.00         9
      Sovrasti

In [7]:
results_df = pd.DataFrame(results).sort_values(
    by='Macro_F1',
    ascending=False
)

results_df

,Model,Accuracy,Macro_F1
1,Decision Tree,0.858333,0.388266
0,Logistic Regression,0.845833,0.305493
2,Random Forest,0.845833,0.305493
3,SVM (RBF),0.845833,0.305493
4,MLP,0.845833,0.305493


## Model comparison
- Tree-based models outperform linear baselines
- Non-linear models capture complex relationships
- Macro-F1 is preferred due to class imbalance